<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/03_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — LSTM Model for Sepsis Prediction

**Pipeline summary**
- Input X : 3-D array `(n_stays, 24, 7)` — 24 hourly time-steps × 7 vital-sign features
- Tabular features : temporal stats, SOFA features, static patient info, lab features, missingness indicators
- Labels : `y_6h`, `y_12h`, `y_24h` (sepsis onset within 6 / 12 / 24 h after the first 24 h window)
- Model : Bidirectional LSTM → Dense head (predicts from time-series)
- Hybrid model : LSTM branch + tabular branch → merged → final prediction


## 1. Mount Drive & Load Preprocessed Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
shutil.copy(
    '/content/drive/MyDrive/gp/preprocessing.py',
    '/content/preprocessing.py'
)

In [ ]:
from preprocessing import (
    run_full_pipeline,
    load_cohort,
    load_vitals_complete,
    load_sofa,
    load_sepsis_labels,
    build_X_array,
    FEATURE_COLS,
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path('/content/drive/MyDrive/mimic_iv_processed')

# Run or reload the full preprocessing pipeline
pipeline = run_full_pipeline(force_rebuild=False)

cohort              = pipeline['cohort']
vitals_complete     = pipeline['vitals_complete']
sofa_df             = pipeline['sofa_df']
suspected_infection = pipeline['suspected_infection']
labels_ordered      = pipeline['labels_ordered']
X                   = pipeline['X']                  # shape (n_stays, 24, 7)
stay_ids_order      = pipeline['stay_ids_order']
label_arrays        = pipeline['label_arrays']

print(f'X shape        : {X.shape}')
print(f'Total stays    : {len(stay_ids_order):,}')
print(f'Label columns  : {labels_ordered.columns.tolist()}')



```
# This is formatted as code
```

## 2. Rebuild Tabular Features (from notebook 02)

In [ ]:
import zipfile
from scipy.stats import linregress

# ---------- Extra labs (Lactate + WBC) ----------
extra_labs = pd.read_csv(OUTPUT_DIR / 'extra_labs_filtered.csv')
extra_labs['charttime'] = pd.to_datetime(extra_labs['charttime'])
extra_labs = extra_labs.dropna(subset=['valuenum'])

extra_lab_itemid_to_name = {50813: 'lactate', 52442: 'lactate', 51301: 'wbc'}
extra_labs['lab_name'] = extra_labs['itemid'].map(extra_lab_itemid_to_name)
extra_labs['charttime_hour'] = extra_labs['charttime'].dt.floor('h')

extra_labs_hourly = (
    extra_labs
    .groupby(['stay_id', 'charttime_hour', 'lab_name'])['valuenum']
    .mean().reset_index()
)

extra_labs_wide = extra_labs_hourly.pivot_table(
    index=['stay_id', 'charttime_hour'],
    columns='lab_name', values='valuenum'
).reset_index()
extra_labs_wide.columns.name = None
extra_labs_wide['charttime_hour'] = pd.to_datetime(extra_labs_wide['charttime_hour'])
extra_labs_wide = extra_labs_wide.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
extra_labs_wide['intime'] = pd.to_datetime(extra_labs_wide['intime'])
extra_labs_wide['hours_since_admit'] = (
    (extra_labs_wide['charttime_hour'] - extra_labs_wide['intime']).dt.total_seconds() / 3600
)
extra_labs_24h = extra_labs_wide[
    (extra_labs_wide['hours_since_admit'] >= 0) &
    (extra_labs_wide['hours_since_admit'] <  24)
].copy()
extra_labs_24h['hour'] = extra_labs_24h['hours_since_admit'].astype(int)

# Forward-fill within each stay
extra_labs_24h = extra_labs_24h.sort_values(['stay_id', 'hour'])
for col in ['lactate', 'wbc']:
    extra_labs_24h[col] = extra_labs_24h.groupby('stay_id')[col].transform(lambda x: x.ffill())

print('extra_labs_24h shape:', extra_labs_24h.shape)

In [ ]:
# ---------- Lab features ----------
def compute_lab_features(extra_labs_24h):
    records = []
    for stay_id, group in extra_labs_24h.groupby('stay_id'):
        row = {'stay_id': stay_id}
        for lab in ['lactate', 'wbc']:
            vals = group[lab].dropna().values
            if len(vals) == 0:
                row[f'{lab}_mean']    = np.nan
                row[f'{lab}_max']     = np.nan
                row[f'{lab}_min']     = np.nan
                row[f'{lab}_first']   = np.nan
                row[f'{lab}_last']    = np.nan
                row[f'{lab}_count']   = 0
                row[f'{lab}_missing'] = 1
            else:
                row[f'{lab}_mean']    = np.nanmean(vals)
                row[f'{lab}_max']     = np.nanmax(vals)
                row[f'{lab}_min']     = np.nanmin(vals)
                row[f'{lab}_first']   = vals[0]
                row[f'{lab}_last']    = vals[-1]
                row[f'{lab}_count']   = len(vals)
                row[f'{lab}_missing'] = 0
        lactate_vals = group['lactate'].dropna().values
        wbc_vals     = group['wbc'].dropna().values
        row['lactate_elevated'] = int(any(lactate_vals > 2.0))
        row['lactate_critical'] = int(any(lactate_vals > 4.0))
        row['wbc_high']         = int(any(wbc_vals > 12.0))
        row['wbc_low']          = int(any(wbc_vals < 4.0))
        row['wbc_abnormal']     = int(row['wbc_high'] or row['wbc_low'])
        records.append(row)
    return pd.DataFrame(records)

lab_features = compute_lab_features(extra_labs_24h)
print('Lab features shape:', lab_features.shape)

In [ ]:
# ---------- Temporal vital-sign features ----------
def compute_temporal_features(vitals_complete, feature_cols):
    records = []
    for stay_id, group in vitals_complete.groupby('stay_id'):
        group = group.sort_values('hour')
        row   = {'stay_id': stay_id}
        for col in feature_cols:
            vals = group[col].values.astype(float)
            row[f'{col}_mean']  = np.nanmean(vals)
            row[f'{col}_std']   = np.nanstd(vals)
            row[f'{col}_min']   = np.nanmin(vals)
            row[f'{col}_max']   = np.nanmax(vals)
            row[f'{col}_last']  = vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals)
            mask = ~np.isnan(vals)
            if mask.sum() >= 2:
                slope, *_ = linregress(np.arange(24)[mask], vals[mask])
                row[f'{col}_slope'] = slope
            else:
                row[f'{col}_slope'] = np.nan
        records.append(row)
    return pd.DataFrame(records)

print('Computing temporal features...')
temporal_features = compute_temporal_features(vitals_complete, FEATURE_COLS)
print(f'Temporal features shape: {temporal_features.shape}')

In [ ]:
# ---------- SOFA features ----------
def compute_sofa_features(sofa_df, cohort):
    sofa = sofa_df.copy()
    sofa['charttime_hour'] = pd.to_datetime(sofa['charttime_hour'])
    sofa = sofa.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
    sofa['intime'] = pd.to_datetime(sofa['intime'])
    sofa['hours_since_admit'] = (
        (sofa['charttime_hour'] - sofa['intime']).dt.total_seconds() / 3600
    )
    sofa_24h = sofa[
        (sofa['hours_since_admit'] >= 0) &
        (sofa['hours_since_admit'] <  24)
    ].copy()
    sofa_24h['hour'] = sofa_24h['hours_since_admit'].astype(int)
    records = []
    for stay_id, group in sofa_24h.groupby('stay_id'):
        group = group.sort_values('hour')
        vals  = group['sofa_total'].values.astype(float)
        h     = group['hour'].values
        mask  = ~np.isnan(vals)
        slope = np.nan
        if mask.sum() >= 2:
            slope, *_ = linregress(h[mask], vals[mask])
        records.append({
            'stay_id':        stay_id,
            'sofa_max_24h':   np.nanmax(vals)  if mask.any() else np.nan,
            'sofa_mean_24h':  np.nanmean(vals) if mask.any() else np.nan,
            'sofa_last_24h':  vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals),
            'sofa_slope_24h': slope,
            'sofa_hours_ge2': int((vals >= 2).sum()),
        })
    return pd.DataFrame(records)

print('Computing SOFA features...')
sofa_features = compute_sofa_features(sofa_df, cohort)
print(f'SOFA features shape: {sofa_features.shape}')

In [ ]:
# ---------- Static patient features ----------
def compute_static_features(cohort):
    static = cohort[[
        'stay_id', 'anchor_age', 'gender',
        'admission_type', 'admission_location', 'icu_los_hours'
    ]].copy()
    static['gender_male'] = (static['gender'].str.upper() == 'M').astype(int)
    admission_dummies = pd.get_dummies(
        static['admission_type'].str.lower().str.replace(' ', '_'),
        prefix='adm_type'
    ).astype(int)
    static = pd.concat([static, admission_dummies], axis=1)
    static = static.drop(columns=['gender', 'admission_type', 'admission_location'])
    return static

static_features = compute_static_features(cohort)
print(f'Static features shape: {static_features.shape}')

In [ ]:
# ---------- Missingness indicators ----------
def compute_missingness_features(vitals_complete, feature_cols):
    records = []
    for stay_id, group in vitals_complete.groupby('stay_id'):
        row = {'stay_id': stay_id}
        for col in feature_cols:
            row[f'{col}_missing_frac'] = group[col].isna().sum() / len(group)
        records.append(row)
    return pd.DataFrame(records)

print('Computing missingness features...')
missingness_features = compute_missingness_features(vitals_complete, FEATURE_COLS)
print(f'Missingness features shape: {missingness_features.shape}')

In [ ]:
# ---------- Combine all features ----------
all_features = (
    pd.DataFrame({'stay_id': stay_ids_order})
    .merge(temporal_features,    on='stay_id', how='left')
    .merge(sofa_features,        on='stay_id', how='left')
    .merge(static_features,      on='stay_id', how='left')
    .merge(missingness_features, on='stay_id', how='left')
    .merge(lab_features,         on='stay_id', how='left')
    .merge(labels_ordered,       on='stay_id', how='left')
)

print(f'Final feature table shape : {all_features.shape}')
print(f'Missing values (top 10)   :')
print(all_features.isnull().sum().sort_values(ascending=False).head(10))

## 3. Prepare Labels & Eligibility Masks

In [ ]:
# Extract the three prediction horizons and their eligibility masks
y_6h  = all_features['y_6h'].fillna(0).astype(int).values
y_12h = all_features['y_12h'].fillna(0).astype(int).values
y_24h = all_features['y_24h'].fillna(0).astype(int).values

eligible_6h  = all_features['eligible_6h'].fillna(False).astype(bool).values
eligible_12h = all_features['eligible_12h'].fillna(False).astype(bool).values
eligible_24h = all_features['eligible_24h'].fillna(False).astype(bool).values

for name, y, elig in [
    ('6h',  y_6h,  eligible_6h),
    ('12h', y_12h, eligible_12h),
    ('24h', y_24h, eligible_24h),
]:
    y_sub = y[elig]
    print(f'y_{name} — eligible: {elig.sum():,}  |  positives: {y_sub.sum():,}  |  rate: {y_sub.mean():.4f}')

## 4. Build Tabular Feature Matrix & Scale

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Columns to exclude from the tabular feature matrix
DROP_COLS = [
    'stay_id',
    'y_6h', 'y_12h', 'y_24h',
    'eligible_6h', 'eligible_12h', 'eligible_24h',
    'sepsis_onset_time', 'onset_hour', 'icu_los_hours'
]

tabular_cols = [c for c in all_features.columns if c not in DROP_COLS]
print(f'Number of tabular features: {len(tabular_cols)}')

T_raw = all_features[tabular_cols].values.astype(np.float32)

# Impute then scale — fitted on training set only (applied after split)
print('Tabular matrix shape:', T_raw.shape)

## 5. Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# ── We work with the 24-h horizon throughout.
# ── Change TARGET_LABEL to 'y_6h' or 'y_12h' to train other horizons.
TARGET_LABEL = 'y_24h'

y_target  = y_24h
elig_mask = eligible_24h

# Keep only eligible stays
X_elig = X[elig_mask]           # (n_elig, 24, 7)
T_elig = T_raw[elig_mask]       # (n_elig, n_tab_feats)
y_elig = y_target[elig_mask]    # (n_elig,)

print(f'Eligible stays : {elig_mask.sum():,}')
print(f'Positive rate  : {y_elig.mean():.4f}')

# 70 / 15 / 15 stratified split
idx = np.arange(len(y_elig))

idx_trainval, idx_test = train_test_split(
    idx, test_size=0.15, random_state=42, stratify=y_elig
)
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=y_elig[idx_trainval]
)

X_train, X_val, X_test = X_elig[idx_train], X_elig[idx_val], X_elig[idx_test]
T_train, T_val, T_test = T_elig[idx_train], T_elig[idx_val], T_elig[idx_test]
y_train, y_val, y_test = y_elig[idx_train], y_elig[idx_val], y_elig[idx_test]

print(f'\nTrain : {len(y_train):,}  |  positives: {y_train.sum():,}  |  rate: {y_train.mean():.4f}')
print(f'Val   : {len(y_val):,}  |  positives: {y_val.sum():,}  |  rate: {y_val.mean():.4f}')
print(f'Test  : {len(y_test):,}  |  positives: {y_test.sum():,}  |  rate: {y_test.mean():.4f}')

In [ ]:
# Scale time-series features (per-feature mean/std fitted on training set)
# X shape: (n, 24, 7)
n_train, T_steps, n_feats = X_train.shape

X_train_2d = X_train.reshape(-1, n_feats)
ts_scaler  = StandardScaler()
X_train_2d = ts_scaler.fit_transform(X_train_2d)
X_train    = X_train_2d.reshape(n_train, T_steps, n_feats).astype(np.float32)

X_val  = ts_scaler.transform(X_val.reshape(-1, n_feats)).reshape(len(X_val),  T_steps, n_feats).astype(np.float32)
X_test = ts_scaler.transform(X_test.reshape(-1, n_feats)).reshape(len(X_test), T_steps, n_feats).astype(np.float32)

# Impute then scale tabular features (fitted on training set)
tab_imputer = SimpleImputer(strategy='median')
T_train = tab_imputer.fit_transform(T_train).astype(np.float32)
T_val   = tab_imputer.transform(T_val).astype(np.float32)
T_test  = tab_imputer.transform(T_test).astype(np.float32)

tab_scaler = StandardScaler()
T_train = tab_scaler.fit_transform(T_train).astype(np.float32)
T_val   = tab_scaler.transform(T_val).astype(np.float32)
T_test  = tab_scaler.transform(T_test).astype(np.float32)

print('X_train:', X_train.shape)
print('T_train:', T_train.shape)

## 6. Handle Class Imbalance

1.   List item
2.   List item



In [ ]:
# Compute class weight to handle imbalance without oversampling
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f'Class weights: {class_weight_dict}')

## 7. Build the Hybrid LSTM + Tabular Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
def build_hybrid_model(ts_steps, ts_feats, tab_feats, lstm_units=64, dropout_rate=0.3):
    """
    Hybrid model:
      - Branch 1 : Bidirectional LSTM on time-series vitals (24 h x 7 features)
      - Branch 2 : Dense MLP on tabular features
      - Merge    : Concatenate both branches -> final Dense -> sigmoid
    """

    # ── Branch 1: Time-series (LSTM) ──────────────────────────────────
    ts_input = Input(shape=(ts_steps, ts_feats), name='ts_input')

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True, dropout=dropout_rate)
    )(ts_input)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units // 2, return_sequences=False, dropout=dropout_rate)
    )(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(32, activation='relu')(x)
    lstm_out = layers.Dropout(dropout_rate)(x)

    # ── Branch 2: Tabular (Dense MLP) ─────────────────────────────────
    tab_input = Input(shape=(tab_feats,), name='tab_input')

    t = layers.Dense(128, activation='relu')(tab_input)
    t = layers.BatchNormalization()(t)
    t = layers.Dropout(dropout_rate)(t)
    t = layers.Dense(64, activation='relu')(t)
    t = layers.BatchNormalization()(t)
    tab_out = layers.Dropout(dropout_rate)(t)

    # ── Merge ──────────────────────────────────────────────────────────
    merged = layers.Concatenate()([lstm_out, tab_out])
    merged = layers.Dense(64, activation='relu')(merged)
    merged = layers.Dropout(dropout_rate)(merged)
    merged = layers.Dense(32, activation='relu')(merged)
    output = layers.Dense(1, activation='sigmoid', name='output')(merged)

    model = Model(inputs=[ts_input, tab_input], outputs=output)
    return model


model = build_hybrid_model(
    ts_steps=X_train.shape[1],     # 24
    ts_feats=X_train.shape[2],     # 7
    tab_feats=T_train.shape[1],    # number of tabular features
    lstm_units=64,
    dropout_rate=0.3
)

model.summary()

## 8. Compile & Train

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='auroc', curve='ROC'),
        tf.keras.metrics.AUC(name='auprc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

In [ ]:
CHECKPOINT_PATH = str(OUTPUT_DIR / 'best_lstm_model.keras')

callbacks = [
    EarlyStopping(
        monitor='val_auprc',
        patience=8,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_auprc',
        factor=0.5,
        patience=4,
        mode='max',
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_auprc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
]

history = model.fit(
    x={'ts_input': X_train, 'tab_input': T_train},
    y=y_train,
    validation_data=(
        {'ts_input': X_val, 'tab_input': T_val},
        y_val
    ),
    epochs=50,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = [
    ('loss',   'Loss'),
    ('auroc',  'AUROC'),
    ('auprc',  'AUPRC'),
]

for ax, (metric, title) in zip(axes, metrics_to_plot):
    ax.plot(history.history[metric],     label='Train')
    ax.plot(history.history[f'val_{metric}'], label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True)

plt.suptitle(f'Training Curves — {TARGET_LABEL} horizon', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()
print('Training curves saved.')

## 10. Evaluate on Test Set

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

# Get predicted probabilities on test set
y_prob = model.predict(
    {'ts_input': X_test, 'tab_input': T_test},
    batch_size=256, verbose=0
).flatten()

# Default threshold = 0.5
y_pred = (y_prob >= 0.5).astype(int)

auroc = roc_auc_score(y_test, y_prob)
auprc = average_precision_score(y_test, y_prob)

print('=' * 50)
print(f'Target horizon : {TARGET_LABEL}')
print(f'Test AUROC     : {auroc:.4f}')
print(f'Test AUPRC     : {auprc:.4f}')
print('=' * 50)
print('\nClassification Report (threshold = 0.5):')
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(f'  TN={tn}  FP={fp}')
print(f'  FN={fn}  TP={tp}')
print(f'\nSensitivity (Recall) : {tp / (tp + fn):.4f}')
print(f'Specificity          : {tn / (tn + fp):.4f}')
print(f'PPV (Precision)      : {tp / (tp + fp):.4f}')
print(f'NPV                  : {tn / (tn + fn):.4f}')

## 11. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, lw=2, label=f'AUROC = {auroc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve — {TARGET_LABEL}')
axes[0].legend()
axes[0].grid(True)

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
baseline = y_test.mean()
axes[1].plot(rec, prec, lw=2, label=f'AUPRC = {auprc:.3f}')
axes[1].axhline(y=baseline, color='k', linestyle='--', lw=1, label=f'Baseline = {baseline:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve — {TARGET_LABEL}')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'roc_pr_curves.png'), dpi=150)
plt.show()
print('ROC & PR curves saved.')

## 12. Optimal Threshold (Youden's J)

In [ ]:
# Find the threshold that maximises sensitivity + specificity
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
youden_idx       = np.argmax(tpr - fpr)
optimal_threshold = thresholds[youden_idx]

print(f'Optimal threshold (Youden J): {optimal_threshold:.4f}')
print(f'  Sensitivity at threshold  : {tpr[youden_idx]:.4f}')
print(f'  Specificity at threshold  : {1 - fpr[youden_idx]:.4f}')

y_pred_opt = (y_prob >= optimal_threshold).astype(int)
print('\nClassification Report at optimal threshold:')
print(classification_report(y_test, y_pred_opt, digits=4))

## 13. Train All Three Horizons & Compare

In [ ]:
horizon_results = {}

HORIZONS = [
    ('y_6h',  eligible_6h,  y_6h),
    ('y_12h', eligible_12h, y_12h),
    ('y_24h', eligible_24h, y_24h),
]

for horizon_name, elig, y_full in HORIZONS:
    print(f'\n{"="*55}')
    print(f'Training horizon: {horizon_name}')
    print(f'{"="*55}')

    X_e = X[elig]
    T_e = T_raw[elig]
    y_e = y_full[elig]

    print(f'  Eligible: {elig.sum():,}  |  positive rate: {y_e.mean():.4f}')

    idx_e = np.arange(len(y_e))
    idx_tv, idx_te = train_test_split(idx_e, test_size=0.15, random_state=42, stratify=y_e)
    idx_tr, idx_va = train_test_split(idx_tv, test_size=0.15/0.85, random_state=42, stratify=y_e[idx_tv])

    Xtr, Xva, Xte = X_e[idx_tr], X_e[idx_va], X_e[idx_te]
    Ttr, Tva, Tte = T_e[idx_tr], T_e[idx_va], T_e[idx_te]
    ytr, yva, yte = y_e[idx_tr], y_e[idx_va], y_e[idx_te]

    # Scale time-series
    n_tr, ts, nf = Xtr.shape
    sc_ts = StandardScaler()
    Xtr = sc_ts.fit_transform(Xtr.reshape(-1, nf)).reshape(n_tr, ts, nf).astype(np.float32)
    Xva = sc_ts.transform(Xva.reshape(-1, nf)).reshape(len(Xva), ts, nf).astype(np.float32)
    Xte = sc_ts.transform(Xte.reshape(-1, nf)).reshape(len(Xte), ts, nf).astype(np.float32)

    # Scale tabular
    imp = SimpleImputer(strategy='median')
    Ttr = imp.fit_transform(Ttr).astype(np.float32)
    Tva = imp.transform(Tva).astype(np.float32)
    Tte = imp.transform(Tte).astype(np.float32)
    sc_tab = StandardScaler()
    Ttr = sc_tab.fit_transform(Ttr).astype(np.float32)
    Tva = sc_tab.transform(Tva).astype(np.float32)
    Tte = sc_tab.transform(Tte).astype(np.float32)

    cw = compute_class_weight('balanced', classes=np.array([0,1]), y=ytr)
    cw_dict = {0: cw[0], 1: cw[1]}

    m = build_hybrid_model(ts_steps=ts, ts_feats=nf, tab_feats=Ttr.shape[1])
    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auprc', curve='PR')]
    )
    cb = [
        EarlyStopping(monitor='val_auprc', patience=8, mode='max', restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_auprc', factor=0.5, patience=4, mode='max', min_lr=1e-6),
    ]
    m.fit(
        x={'ts_input': Xtr, 'tab_input': Ttr}, y=ytr,
        validation_data=({'ts_input': Xva, 'tab_input': Tva}, yva),
        epochs=50, batch_size=256, class_weight=cw_dict,
        callbacks=cb, verbose=0
    )

    yp = m.predict({'ts_input': Xte, 'tab_input': Tte}, batch_size=256, verbose=0).flatten()
    auroc_h = roc_auc_score(yte, yp)
    auprc_h = average_precision_score(yte, yp)

    horizon_results[horizon_name] = {
        'auroc': auroc_h, 'auprc': auprc_h,
        'n_test': len(yte), 'pos_rate': yte.mean()
    }
    print(f'  AUROC: {auroc_h:.4f}  |  AUPRC: {auprc_h:.4f}')

print('\n--- Summary ---')
for h, r in horizon_results.items():
    print(f'{h:6s} | AUROC={r["auroc"]:.4f} | AUPRC={r["auprc"]:.4f} | n_test={r["n_test"]:,} | pos_rate={r["pos_rate"]:.4f}')

## 14. Save Model & Scalers

In [ ]:
import pickle

# Save the best model (24h horizon, already restored from checkpoint)
model.save(str(OUTPUT_DIR / 'lstm_sepsis_24h.keras'))
print('Model saved.')

# Save scalers and imputer so they can be reloaded for inference
with open(str(OUTPUT_DIR / 'ts_scaler.pkl'), 'wb') as f:
    pickle.dump(ts_scaler, f)

with open(str(OUTPUT_DIR / 'tab_imputer.pkl'), 'wb') as f:
    pickle.dump(tab_imputer, f)

with open(str(OUTPUT_DIR / 'tab_scaler.pkl'), 'wb') as f:
    pickle.dump(tab_scaler, f)

# Save tabular column names so we can reconstruct T correctly at inference
with open(str(OUTPUT_DIR / 'tabular_cols.pkl'), 'wb') as f:
    pickle.dump(tabular_cols, f)

print('Scalers and column list saved.')

## 15. Reload & Quick Sanity Check

In [ ]:
loaded_model = tf.keras.models.load_model(str(OUTPUT_DIR / 'lstm_sepsis_24h.keras'))

y_prob_check = loaded_model.predict(
    {'ts_input': X_test[:10], 'tab_input': T_test[:10]},
    verbose=0
).flatten()

print('Predicted probabilities (first 10 test samples):')
for i, (prob, true) in enumerate(zip(y_prob_check, y_test[:10])):
    print(f'  Sample {i+1:2d} | prob={prob:.4f} | true={true}')